In [2]:
import sys
!{sys.executable} -m pip install langchain langchain-huggingface langchain-community langgraph chromadb pypdf

In [3]:
import sys
!{sys.executable} -m pip install sentence-transformers

In [ ]:
import os
from typing import TypedDict, Optional, List, Any
from langchain_core.documents import Document
from langchain_core.language_models.llms import LLM
from langchain_core.prompts import PromptTemplate
from langgraph.graph import StateGraph, START, END

print("Initializing Offline Enterprise Simulators...")

# ==========================================
# 1. OFFLINE MOCK LLM
# ==========================================
class OfflineMockLLM(LLM):
    @property
    def _llm_type(self) -> str:
        return "offline_mock_llm"

    def _call(self, prompt: str, stop: Optional[List[str]] = None, **kwargs: Any) -> str:
        if "Analyze the user's question" in prompt:
            if "furious" in prompt.lower():
                return "ESCALATE"
            return "ANSWER"

        if "Use the following context to answer" in prompt:
            return (
                "Based on the internship document, the objective is to design "
                "and implement a Retrieval-Augmented Generation (RAG) system "
                "using LangGraph that processes PDFs, routes intents, and "
                "supports Human-in-the-Loop (HITL) escalation."
            )

        return "Default simulated response."

llm = OfflineMockLLM()

# ==========================================
# 2. MOCK RETRIEVER
# ==========================================
class EnterpriseMockRetriever:
    def invoke(self, question: str):
        print("   [Database Simulator]: Retrieving context...")
        simulated_context = """
        RAG INTERNSHIP PROJECT - Customer Support Assistant.
        Objective: Design a Retrieval-Augmented Generation (RAG) system using LangGraph.
        Features: Processes PDFs, routes intents, and supports Human-in-the-Loop (HITL) escalation.
        """
        return [Document(page_content=simulated_context)]

retriever = EnterpriseMockRetriever()

# ==========================================
# 3. STATE
# ==========================================
class AgentState(TypedDict):
    question: str
    context: str
    intent: str
    answer: str

# ==========================================
# 4. NODES
# ==========================================
def retrieve_node(state: AgentState):
    docs = retriever.invoke(state["question"])
    context = "\n".join([d.page_content for d in docs])
    return {"context": context}

def intent_routing_node(state: AgentState):
    prompt = PromptTemplate.from_template(
        "Analyze the user's question: '{question}'. "
        "Reply with 'ESCALATE' or 'ANSWER'."
    )
    chain = prompt | llm
    intent = chain.invoke({"question": state["question"]}).strip().upper()
    return {"intent": intent}

def generate_answer_node(state: AgentState):
    prompt = PromptTemplate.from_template(
        "Use the following context:\n{context}\nQuestion: {question}\nAnswer:"
    )
    chain = prompt | llm
    answer = chain.invoke({
        "context": state["context"],
        "question": state["question"]
    })
    return {"answer": answer}

def human_in_the_loop_node(state: AgentState):
    print(f"\n⚠️ ESCALATION TRIGGERED: {state['question']}")
    human_response = input("HUMAN AGENT RESPONSE: ")
    return {"answer": f"[Support Agent]: {human_response}"}

# ==========================================
# 5. GRAPH
# ==========================================
workflow = StateGraph(AgentState)

workflow.add_node("retrieve", retrieve_node)
workflow.add_node("analyze_intent", intent_routing_node)
workflow.add_node("generate", generate_answer_node)
workflow.add_node("hitl", human_in_the_loop_node)

workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "analyze_intent")

def route_logic(state: AgentState):
    return "hitl" if "ESCALATE" in state["intent"] else "generate"

workflow.add_conditional_edges("analyze_intent", route_logic)
workflow.add_edge("generate", END)
workflow.add_edge("hitl", END)

app = workflow.compile()

# ==========================================
# 6. RUN
# ==========================================
print("\n--- TEST 1 ---")
res1 = app.invoke({"question": "What is the main objective of this internship project?"})
print("Output:", res1["answer"])

print("\n--- TEST 2 ---")
res2 = app.invoke({"question": "I am furious! Let me talk to a human agent immediately!"})
print("Output:", res2["answer"])

print("\n✅ Done")

Initializing Offline Enterprise Simulators...
Building LangGraph Workflow...

--- TEST 1: Standard Query ---
   [Database Simulator]: Retrieving context from vector space...
Final Output: Based on the internship document, the objective is to design and implement a Retrieval-Augmented Generation (RAG) system using LangGraph that processes PDFs, routes intents, and supports Human-in-the-Loop (HITL) escalation.

--- TEST 2: Escalation Query ---
   [Database Simulator]: Retrieving context from vector space...

⚠️ ESCALATION TRIGGERED FOR QUERY: I am furious! Let me talk to a human agent immediately!


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
